# 5_Dataset_Preparation

This notebook finalizes the preprocessing of the merged SEO dataset by:

- **Cleaning and transforming** categorical features with many distinct values (e.g., HQ_State, New_Issues_HiTech_Group, Regulation_Types, Security_Type_This_Mkt) using one-hot encoding or logical grouping.

- **Encoding ordinal features** such as S&P_Quality_Ranking into numeric rankings.

- **Handling missing data** by:

    - Dropping columns with more than 45% missing values.

    - Removing rows with any remaining nulls.

- **Exporting** the cleaned dataset in both CSV and Parquet formats for modeling readiness.

The output dataset is now fully encoded, free from high-missing-value columns, and suitable for regression or classification modeling.



In [ ]:
import pandas as pd

# Step 1: Load the cleaned and enriched SEO dataset
file_path = r'data\Updated SEO ISSUES Dataset\4SEO_Merged_For_Cleaning.parquet'
df = pd.read_parquet(file_path)

# Step 2: Preview first few rows and dataset structure
print(df.head())
df.info()

# Step 3: Check proportion of missing values in each column
missing_proportion = df.isnull().mean()
missing_proportion_sorted = missing_proportion.sort_values(ascending=False)

# Display only columns that contain missing values
missing_proportion_sorted[missing_proportion_sorted > 0]


In [ ]:
# Step 4: Examine the unique values in the HiTech group column
unique_values = df['New_Issues_HiTech_Group'].dropna().unique()
print(unique_values)
print("Total unique values:", len(unique_values))

# Step 5: Split semi-colon-separated values and flatten the list
all_groups = df['New_Issues_HiTech_Group'].dropna().str.split(';').sum()
unique_groups = set([x.strip() for x in all_groups])
print(unique_groups)
print("Total individual group types:", len(unique_groups))

# Step 6: Manually selected 6 primary categories to create dummy variables
categories = ["Computer Equipment", "Electronics", "Communications", 
              "General Technology", "Non-Hitech", "Biotechnology"]

# Step 7: Create binary indicator (dummy) columns for each of the 6 categories
for category in categories:
    df[category + "_dummy"] = df['New_Issues_HiTech_Group'].fillna('').apply(lambda x: 1 if category in x else 0)

# Step 8: Preview the newly created dummy columns
df[[cat + "_dummy" for cat in categories]]


In [ ]:
# Step 9: Explore distinct regulation types
unique_values = df['Regulation_Types'].dropna().unique()
print(unique_values)
print("Total unique values:", len(unique_values))

# Step 10: Extract and count distinct regulation labels
all_groups = df['Regulation_Types'].dropna().str.split(';').sum()
unique_groups = set([x.strip() for x in all_groups])
print(unique_groups)
print("Total individual group types:", len(unique_groups))

# Step 11: Select top 12 categories based on domain knowledge
categories = [
    "Registration Rights", "Shelf Filing", "JOBS Act", 
    "SEC Registered", "SIB Stabilisation", "SEC R", 
    "Reg S", "Reg D", "Rule 144A Eligible", 
    "Shelf Filing/Rule", "Shelf Filing/Rule 415", "Rule 144A Elig"
]

# Step 12: Create binary dummy variables for each selected regulation category
for category in categories:
    df[category + "_dummy"] = df['Regulation_Types'].fillna('').apply(lambda x: 1 if category in x else 0)

# Step 13: Preview regulation dummy variables
print(df[[cat + "_dummy" for cat in categories]].head())


In [ ]:
# Step 14: Review distinct values
unique_values = df['Security_Type_This_Mkt'].dropna().unique()
print(unique_values)
print("Total unique values:", len(unique_values))

# Step 15: Split and count individual values
all_groups = df['Security_Type_This_Mkt'].dropna().str.split(';').sum()
unique_groups = set([x.strip() for x in all_groups])
print(unique_groups)
print("Total individual group types:", len(unique_groups))

# Step 16: Define mapping from granular types to 6 high-level categories
type_mapping = {
    "Common Shares": [...],
    "Preferred Shares": [...],
    "Depositary Receipts": [...],
    "Units and Trusts": [...],
    "Partnership/LLC Interests": [...],
    "Other": [...]
}

# Step 17: Function to group security types into categories
def map_security_type(raw):
    for group, options in type_mapping.items():
        if raw in options:
            return group
    return "Other"

# Step 18: Apply mapping function to create new grouped column
df["Security_Type_Grouped"] = df["Security_Type_This_Mkt"].apply(map_security_type)

# Step 19: One-hot encode the new grouped security type variable (k-1 dummies)
df = pd.get_dummies(
    df,
    columns=["Security_Type_Grouped"],
    prefix="SecType",
    drop_first=True,     # Avoid multicollinearity by dropping one category
    dtype="int8"
)

# Step 20: Verify the dummy columns for Security Type
print([c for c in df.columns if c.startswith("SecType_")])
print(df.filter(like="SecType_").head())


In [ ]:
# Get unique values in HQ_State (e.g., "CA", "NY", etc.)
unique_values = df['HQ_State'].dropna().unique()
print(unique_values)
print("Total unique values:", len(unique_values))

# Some values might be concatenated with semicolons (e.g., "CA;NY"), so we split and flatten
all_groups = df['HQ_State'].dropna().str.split(';').sum()
unique_groups = set([x.strip() for x in all_groups])
print(unique_groups)
print("Total individual group types:", len(unique_groups))

# One-hot encode HQ_State with k‑1 dummies (drop_first=True)
df = pd.get_dummies(df, columns=['HQ_State'], prefix='State', drop_first=True, dtype='int8')

# Sanity check: list new dummy columns
state_dummies = [c for c in df.columns if c.startswith('State_')]
print(f"{len(state_dummies)} dummy columns created:", state_dummies[:10], '...')
print(df[state_dummies].head())


In [ ]:
# Create 11 dummy columns for Offer_Month (excluding base month = January)
base_month = 1
for m in range(1, 13):
    if m == base_month:
        continue
    df[f"Offer_Month_{m}"] = (df['Offer_Month'] == m).astype(int)

# Sanity check: view newly created monthly dummies
print(df.filter(like="Offer_Month_").head())


In [ ]:
# Group exchange into 3 categories
df['New_Issues_Primary_Exchange_Grouped'] = df['New_Issues_Primary_Exchange'].apply(
    lambda x: 'Nasdaq' if x == 'Nasdaq' 
              else ('NYSE' if x == 'New York Stock Exchange' else 'Others')
)

# One-hot encode with k‑1 dummy (drop Nasdaq)
df = pd.get_dummies(df, columns=['New_Issues_Primary_Exchange_Grouped'], 
                    prefix='Primary_Exchange', drop_first=True, dtype='int8')

# Check dummy columns created
exchange_dummies = [c for c in df.columns if c.startswith('Primary_Exchange_')]
print(f"{len(exchange_dummies)} dummy columns created:", exchange_dummies)
print(df[exchange_dummies].head())


In [ ]:
# Convert VC-backed flag to boolean
df['VC_Backed_IPO_Issue_Flag'] = df['VC_Backed_IPO_Issue_Flag'].astype('bool')
print(df['VC_Backed_IPO_Issue_Flag'].dtype)
print(df['VC_Backed_IPO_Issue_Flag'].head())

# One-hot encode Marketplace (drop alphabetically first: "U.S. Private")
df = pd.get_dummies(df, columns=['Marketplace'], prefix='Marketplace', 
                    drop_first=True, dtype='int8')

# One-hot encode Market Area (drop "Domestic")
df = pd.get_dummies(df, columns=['Market_Area'], prefix='Market_Area', 
                    drop_first=True, dtype='int8')

# One-hot encode Share Type Offered (drop first category)
df = pd.get_dummies(df, columns=['Share_Type_Offered'], prefix='Share_Type', 
                    drop_first=True, dtype='int8')


In [ ]:
# View distinct S&P rankings
unique_values = df['S_P_Quality_Ranking'].dropna().unique()
print(unique_values)
print("Total unique values:", len(unique_values))

# Some entries might be semicolon-separated
all_groups = df['S_P_Quality_Ranking'].dropna().str.split(';').sum()
unique_groups = set([x.strip() for x in all_groups])
print(unique_groups)
print("Total individual group types:", len(unique_groups))

# Define rank order manually
ranking_order = {'D': 1, 'C': 2, 'B-': 3, 'B': 4, 'B+': 5, 
                 'A-': 6, 'A': 7, 'A+': 8}

# Map to numerical values, fill NaNs with 0, and convert to int
df['S_P_Quality_Ranking_Num'] = df['S_P_Quality_Ranking'].astype(object).map(ranking_order).fillna(0).astype('int8')

# Check result
print(df['S_P_Quality_Ranking_Num'].value_counts().sort_index())


In [ ]:
# Calculate percent of missing values
missing_percent = df.isnull().mean()

# Find columns with more than 45% missing
cols_to_drop = missing_percent[missing_percent > 0.45].index.tolist()
print(f"Columns to be dropped ({len(cols_to_drop)}):", cols_to_drop)

# Drop them
df = df.drop(columns=cols_to_drop)

# Print summary
print(f"Total columns before: {df.shape[1] + len(cols_to_drop)}")
print(f"Total columns dropped: {len(cols_to_drop)}")
print(f"Total columns remaining: {df.shape[1]}")


In [ ]:
# Drop all remaining rows with missing values
df = df.dropna()

# Report number of rows remaining
print(f"Total rows remaining after dropping missing rows: {df.shape[0]}")

# Export to CSV and Parquet
df.to_csv('df_exported.csv', index=False)
df.to_parquet('df_exported.parquet', index=False)
